In [1]:

!pip install pandas numpy scikit-learn joblib


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

In [3]:
# ===============================
# 3️⃣ Generate synthetic dataset
# ===============================
np.random.seed(42)

# Options user can select
states = ['Delhi','Maharashtra','Karnataka','Tamil Nadu','West Bengal']
court_types = ['District Court','High Court','Sessions Court','Supreme Court']
case_types = ['Criminal','Civil','Family','Corporate','Constitutional']

cases_data = []
for i in range(2000):
    state = np.random.choice(states)
    court_type = np.random.choice(court_types)
    case_type = np.random.choice(case_types)
    judge_rating = round(np.random.uniform(5, 10), 1)
    complexity = round(np.random.uniform(1.5, 9.5), 1)
    hearings = np.random.randint(1, 20)
    public_interest = np.random.choice(['Yes','No'], p=[0.1,0.9])
    status = np.random.choice(['Decided','Pending'], p=[0.6,0.4])
    
    case_text = f"The {case_type} case with complexity {complexity} and {hearings} hearings"
    
    cases_data.append({
        'state': state,
        'court_type': court_type,
        'case_type': case_type,
        'judge_rating': judge_rating,
        'complexity': complexity,
        'hearings': hearings,
        'public_interest': public_interest,
        'case_text': case_text,
        'outcome': 1 if status=='Decided' else 0
    })

df_cases = pd.DataFrame(cases_data)
df_cases.head()

,state,court_type,case_type,judge_rating,complexity,hearings,public_interest,case_text,outcome
0,Tamil Nadu,District Court,Family,8.9,6.3,19,Yes,The Family case with complexity 6.3 and 19 hea...,1
1,West Bengal,Supreme Court,Family,5.1,9.3,12,No,The Family case with complexity 9.3 and 12 hea...,1
2,Tamil Nadu,District Court,Criminal,6.5,5.7,12,Yes,The Criminal case with complexity 5.7 and 12 h...,1
3,Maharashtra,Supreme Court,Corporate,9.9,3.4,15,No,The Corporate case with complexity 3.4 and 15 ...,1
4,Tamil Nadu,Supreme Court,Criminal,5.2,6.4,9,Yes,The Criminal case with complexity 6.4 and 9 he...,0


In [4]:
# ===============================
# 4️⃣ Text preprocessing
# ===============================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df_cases['clean_text'] = df_cases['case_text'].apply(clean_text)

In [5]:
# ===============================
# 5️⃣ Split data & vectorize text
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    df_cases['clean_text'], df_cases['outcome'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [6]:
# ===============================
# 6️⃣ Train model
# ===============================
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6075


In [7]:
# ===============================
# 7️⃣ Save model and vectorizer
# ===============================
import os
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(model, "../backend/models/case_outcome_model.pkl")
joblib.dump(vectorizer, "../backend/models/tfidf_vectorizer.pkl")
print("✅ Model and vectorizer saved!")

✅ Model and vectorizer saved!


In [8]:
# ===============================
# 8️⃣ User prediction flow
# ===============================

# Load model/vectorizer
model = joblib.load("../backend/models/case_outcome_model.pkl")
vectorizer = joblib.load("../backend/models/tfidf_vectorizer.pkl")

# Example: user selects only what they know
user_state = "Delhi"
user_court_type = "District Court"
user_case_type = "Criminal"
public_interest = "No"

# Filter dataset for averages
subset = df_cases[
    (df_cases['state']==user_state) &
    (df_cases['court_type']==user_court_type) &
    (df_cases['case_type']==user_case_type)
]

# If subset empty, fallback to full dataset
if len(subset)==0:
    subset = df_cases

avg_complexity = subset['complexity'].mean()
avg_hearings = subset['hearings'].mean()

# Generate text for model
input_text = f"The {user_case_type} case with complexity {avg_complexity:.1f} and {avg_hearings:.0f} hearings"
input_clean = clean_text(input_text)
input_vec = vectorizer.transform([input_clean])

prediction = model.predict(input_vec)[0]
prob = model.predict_proba(input_vec)[0]

print("Prediction:", "Favorable (Decided)" if prediction==1 else "Unfavorable (Pending)")
print("Confidence:", round(max(prob)*100,2), "%")

Prediction: Favorable (Decided)
Confidence: 55.49 %


In [9]:
df_cases.to_csv("../backend/models/cases_data.csv", index=False)